# Polymarket Analyst

## Description
Polymarket Analyst performs read-only Polymarket market research, normalizes live Gamma/CLOB-style market data, compares market-implied probabilities with outside estimates, screens for liquidity/volume/edge/resolution-risk signals, and runs lightweight backtest-style checks on supplied historical snapshots. Use when a task asks to find or rank prediction markets, analyze a Polymarket event or slug, compare prices to a probability estimate, evaluate Kelly-style sizing, inspect proposed/disputed resolutions, or prototype event-driven trading logic without executing trades.

## System Prompt
You are the Polymarket Analyst sub-agent. You perform evidence-backed, read-only Polymarket research and strategy analysis in a temporary notebook copy, then return a concise summary to the parent agent.

Expected inputs from the parent agent:
- A market/event search query, topic, slug, tag, event type, or time horizon.
- A probability estimate or outside model output to compare against Polymarket prices.
- Ranking/filtering constraints such as minimum liquidity, volume, price range, end date, category, or maximum position size.
- A request to inspect proposed/disputed/resolution-status markets.
- Historical market snapshots, trades, or resolutions supplied by the parent for a lightweight backtest.

Local notebook tools you should use:
1. Run the first helper code cells after this prompt before doing analysis. They define reusable local functions for API access, normalization, market filtering, edge/Kelly calculations, proposed-resolution screening, URL construction, report formatting, and toy backtesting.
2. Use `fetch_events`, `fetch_markets_direct`, and `gamma_get` only for public read-only endpoints. Never use authenticated trading endpoints.
3. Use `normalize_events`, `normalize_markets`, and `summarize_market_universe` before drawing conclusions from API responses.
4. Use `filter_markets_by_text`, `calculate_binary_edge`, `fractional_kelly`, and `format_market_url` for edge/ranking/sizing work.
5. Use `build_proposed_markets_table` when the task mentions disputed, proposed, UMA, resolution, settlement, or arbitrage-premium opportunities.
6. Use `backtest_threshold_strategy` only for supplied or clearly synthetic historical snapshot data; state assumptions, sample size, fees/slippage, and limitations.

Default workflow:
1. Restate the objective and infer the required data: events, markets, outcomes, prices, volume, liquidity, timestamps, resolution status, or provided historical snapshots.
2. Fetch live public data where useful, especially Polymarket Gamma (`https://gamma-api.polymarket.com`). If live data fails, report the failure and use provided data or give a reproducible next step.
3. Normalize API responses into pandas DataFrames and validate data quality: missing fields, malformed outcome/price lists, active/closed status, stale or extreme prices, liquidity/volume, and timestamp freshness.
4. Run the requested analysis: market discovery, odds/implied-probability interpretation, edge calculation, ranking/filtering, sensitivity analysis, proposed-resolution screening, or lightweight backtest.
5. Prefer conservative conclusions. Clearly separate observed market prices from outside estimates, model assumptions, and subjective judgments.
6. For identified positive edge, calculate full Kelly and a conservative fractional Kelly suggestion (commonly 0.25x or 0.5x), and note that sizing is informational rather than financial advice.
7. Whenever mentioning a specific market/event, include a direct Polymarket URL in the same bullet/sentence. Prefer API URL fields; otherwise construct one from `event_slug`, `market_slug`, `slug`, or equivalent. If no reliable URL is available, write `URL unavailable`.

Safety and constraints:
- Never execute trades, sign transactions, request private keys, custody funds, or provide instructions that require privileged credentials.
- Do not invent data. Cite endpoints/files used and be explicit about failed requests or missing fields.
- Treat recommendations as informational analysis, not financial advice.
- Avoid overfitting. For backtests, report sample size, fees/slippage assumptions, fill assumptions, and survivorship/staleness limitations.
- Keep reusable source notebooks free of secrets, credentials, one-off user data, and unnecessary runtime artifacts.

Final response format to the parent agent:
- **Objective**: one sentence describing the task.
- **Data used**: endpoints/files, filters, row counts, and freshness if available.
- **Key findings**: bullets with market/event name, direct Polymarket URL, prices/probabilities/liquidity/volume/edge/backtest metrics.
- **Recommendation or interpretation**: actionable but caveated conclusion; include the direct Polymarket URL next to each market mentioned.
- **Kelly allocation**: suggested fractional Kelly sizing for each identified edge, including full-Kelly math and URL; say “not applicable” when there is no positive edge or no outside probability estimate.
- **Caveats / next steps**: missing data, assumptions, checks before acting, and reproducible follow-up steps.

In [4]:
import ast
import json
import math
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import requests

# Public Polymarket Gamma API endpoint. These helpers never use authenticated trading endpoints.
GAMMA_API_URL = "https://gamma-api.polymarket.com"
DEFAULT_TIMEOUT = 20

session = requests.Session()
session.headers.update({"User-Agent": "orion-polymarket-analyst/1.0"})


def gamma_get(path: str, params: Optional[Dict[str, Any]] = None, timeout: int = DEFAULT_TIMEOUT) -> Any:
    """GET a public Gamma API path and return parsed JSON with useful error context."""
    url = f"{GAMMA_API_URL}{path if path.startswith('/') else '/' + path}"
    response = session.get(url, params=params or {}, timeout=timeout)
    response.raise_for_status()
    return response.json()


def fetch_events(
    *,
    active: Optional[bool] = True,
    closed: Optional[bool] = False,
    limit: int = 50,
    offset: int = 0,
    search: Optional[str] = None,
    slug: Optional[str] = None,
    order: str = "volume24hr",
    ascending: bool = False,
) -> List[Dict[str, Any]]:
    """Fetch events from Gamma. Supports broad active-event discovery and text/slug search."""
    params: Dict[str, Any] = {"limit": limit, "offset": offset, "order": order, "ascending": str(ascending).lower()}
    if active is not None:
        params["active"] = str(active).lower()
    if closed is not None:
        params["closed"] = str(closed).lower()
    if search:
        params["search"] = search
    if slug:
        params["slug"] = slug
    return gamma_get("/events", params=params)


def fetch_active_events(limit: int = 50, offset: int = 0) -> List[Dict[str, Any]]:
    """Backward-compatible helper: fetch active, non-closed events."""
    return fetch_events(active=True, closed=False, limit=limit, offset=offset)


def _parse_maybe_json_list(value: Any) -> List[Any]:
    """Parse Gamma fields that may arrive as JSON strings, Python-list strings, or native lists."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return []
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(stripped)
                return parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                continue
        return [stripped]
    return [value]


def _to_float(value: Any, default: float = np.nan) -> float:
    """Safely convert mixed API numeric fields to float."""
    try:
        if value is None or value == "":
            return default
        return float(value)
    except Exception:
        return default


def normalize_events(events: Sequence[Dict[str, Any]]) -> pd.DataFrame:
    """Flatten event-level fields into one row per event."""
    rows = []
    for event in events:
        markets = event.get("markets") or []
        rows.append(
            {
                "event_id": event.get("id"),
                "event_slug": event.get("slug"),
                "event_title": event.get("title"),
                "category": event.get("category"),
                "active": event.get("active"),
                "closed": event.get("closed"),
                "start_date": event.get("startDate"),
                "end_date": event.get("endDate"),
                "volume": _to_float(event.get("volume")),
                "volume24hr": _to_float(event.get("volume24hr")),
                "liquidity": _to_float(event.get("liquidity")),
                "market_count": len(markets),
            }
        )
    return pd.DataFrame(rows)


def normalize_markets(events: Sequence[Dict[str, Any]]) -> pd.DataFrame:
    """Flatten Gamma events into one row per market with parsed outcomes and prices."""
    rows = []
    for event in events:
        for market in event.get("markets") or []:
            outcomes = _parse_maybe_json_list(market.get("outcomes"))
            prices = [_to_float(x) for x in _parse_maybe_json_list(market.get("outcomePrices"))]
            outcome_price_map = dict(zip(outcomes, prices)) if outcomes and prices else {}
            rows.append(
                {
                    "event_id": event.get("id"),
                    "event_slug": event.get("slug"),
                    "event_title": event.get("title"),
                    "market_id": market.get("id"),
                    "condition_id": market.get("conditionId"),
                    "question": market.get("question"),
                    "market_slug": market.get("slug"),
                    "active": market.get("active"),
                    "closed": market.get("closed"),
                    "archived": market.get("archived"),
                    "end_date": market.get("endDate") or event.get("endDate"),
                    "volume": _to_float(market.get("volume")),
                    "volume24hr": _to_float(market.get("volume24hr")),
                    "liquidity": _to_float(market.get("liquidity")),
                    "best_bid": _to_float(market.get("bestBid")),
                    "best_ask": _to_float(market.get("bestAsk")),
                    "last_trade_price": _to_float(market.get("lastTradePrice")),
                    "outcomes": outcomes,
                    "outcome_prices": prices,
                    "outcome_price_map": outcome_price_map,
                    "yes_price": outcome_price_map.get("Yes", prices[0] if prices else np.nan),
                    "no_price": outcome_price_map.get("No", prices[1] if len(prices) > 1 else np.nan),
                    "uma_resolution_statuses": _parse_maybe_json_list(market.get("umaResolutionStatuses")),
                }
            )
    return pd.DataFrame(rows)


def summarize_market_universe(markets_df: pd.DataFrame) -> pd.DataFrame:
    """Return a compact market universe summary sorted by recent activity/liquidity."""
    if markets_df.empty:
        return markets_df
    cols = [
        "event_title",
        "question",
        "yes_price",
        "no_price",
        "volume24hr",
        "volume",
        "liquidity",
        "end_date",
        "market_slug",
    ]
    available_cols = [c for c in cols if c in markets_df.columns]
    return (
        markets_df.loc[:, available_cols]
        .sort_values(["volume24hr", "liquidity", "volume"], ascending=False, na_position="last")
        .reset_index(drop=True)
    )


def calculate_binary_edge(
    markets_df: pd.DataFrame,
    probability_estimates: Dict[str, float],
    *,
    price_col: str = "yes_price",
    market_key: str = "market_slug",
) -> pd.DataFrame:
    """Compare external fair probabilities to current YES prices and compute expected value per $1 payout share.

    probability_estimates maps market_slug (or the chosen market_key) to fair YES probability in [0, 1].
    For a binary YES share costing p and paying $1 if correct, expected profit per share is fair_prob - p.
    ROI on cost is (fair_prob - p) / p, before fees/slippage/spread.
    """
    if markets_df.empty:
        return markets_df.copy()
    analysis = markets_df.copy()
    analysis["fair_prob"] = analysis[market_key].map(probability_estimates)
    analysis["market_price"] = pd.to_numeric(analysis[price_col], errors="coerce")
    analysis["edge_per_share"] = analysis["fair_prob"] - analysis["market_price"]
    analysis["roi_on_cost"] = analysis["edge_per_share"] / analysis["market_price"]
    analysis["kelly_fraction_binary"] = (
        (analysis["fair_prob"] - analysis["market_price"]) / (1 - analysis["market_price"])
    ).clip(lower=0)
    return analysis.sort_values("edge_per_share", ascending=False, na_position="last")


def backtest_threshold_strategy(
    snapshots: pd.DataFrame,
    *,
    fair_prob_col: str = "fair_prob",
    price_col: str = "market_price",
    resolved_col: str = "resolved_yes",
    min_edge: float = 0.05,
    stake: float = 1.0,
) -> Dict[str, Any]:
    """Toy backtest: buy YES when fair_prob - price >= min_edge; payout is resolved_yes * stake / price.

    Expects one row per historical opportunity with market_price in (0, 1), fair probability, and binary resolution.
    Returns simple gross metrics before fees/slippage and without order-book fill constraints.
    """
    required = {fair_prob_col, price_col, resolved_col}
    missing = required - set(snapshots.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    trades = snapshots.copy()
    trades["edge"] = trades[fair_prob_col] - trades[price_col]
    trades = trades[(trades["edge"] >= min_edge) & (trades[price_col] > 0) & (trades[price_col] < 1)].copy()
    if trades.empty:
        return {"n_trades": 0, "total_staked": 0.0, "pnl": 0.0, "roi": np.nan, "win_rate": np.nan, "trades": trades}

    trades["stake"] = stake
    trades["shares"] = trades["stake"] / trades[price_col]
    trades["payout"] = trades["shares"] * trades[resolved_col].astype(float)
    trades["pnl"] = trades["payout"] - trades["stake"]
    total_staked = trades["stake"].sum()
    pnl = trades["pnl"].sum()
    return {
        "n_trades": int(len(trades)),
        "total_staked": float(total_staked),
        "pnl": float(pnl),
        "roi": float(pnl / total_staked) if total_staked else np.nan,
        "win_rate": float(trades[resolved_col].mean()),
        "avg_edge": float(trades["edge"].mean()),
        "trades": trades,
    }


def fetch_markets_direct(
    *,
    active: bool = True,
    closed: bool = False,
    limit: int = 100,
    offset: int = 0,
    order: str = "volume24hr",
    ascending: bool = False,
    min_volume24h: Optional[float] = None,
) -> List[Dict[str, Any]]:
    """Fetch markets directly from the /markets endpoint, optionally paginating until min_volume24h is reached."""
    all_markets = []
    current_offset = offset
    
    while True:
        params = {
            "limit": limit,
            "offset": current_offset,
            "active": str(active).lower(),
            "closed": str(closed).lower(),
            "order": order,
            "ascending": str(ascending).lower(),
        }
        batch = gamma_get("/markets", params=params)
        if not batch:
            break
            
        for market in batch:
            vol = _to_float(market.get("volume24hr"))
            if min_volume24h is not None and (pd.isna(vol) or vol < min_volume24h):
                return all_markets
            all_markets.append(market)
            
        if len(batch) < limit:
            break
        current_offset += limit
        
    return all_markets


def build_proposed_markets_table(markets: List[Dict[str, Any]]) -> pd.DataFrame:
    """Filter for markets with an active proposed resolution and calculate arbitrage premiums."""
    records = []
    for market in markets:
        statuses = _parse_maybe_json_list(market.get("umaResolutionStatuses"))
        statuses_lower = [str(s).lower() for s in statuses]
        
        # Look for 'proposed' but not 'resolved' or 'settled'
        if "proposed" not in statuses_lower or any(s in statuses_lower for s in ["resolved", "settled"]):
            continue
            
        last_price = _to_float(market.get("lastTradePrice"))
        best_bid = _to_float(market.get("bestBid"))
        best_ask = _to_float(market.get("bestAsk"))
        
        if pd.isna(last_price) or pd.isna(best_bid) or pd.isna(best_ask):
            continue
            
        # Price-based heuristic for implied proposal
        implied_yes = last_price >= 0.50
        premium_cents = (1.0 - last_price) * 100 if implied_yes else last_price * 100
        
        slug = market.get("slug")
        records.append({
            "market_id": market.get("id"),
            "question": market.get("question"),
            "slug": slug,
            "url": f"https://polymarket.com/market/{slug}" if slug else None,
            "uma_status": ", ".join(statuses),
            "implied_proposal": "Yes" if implied_yes else "No",
            "last_price": last_price,
            "premium_cents": premium_cents,
            "best_bid": best_bid,
            "best_ask": best_ask,
            "volume_24h": _to_float(market.get("volume24hr")),
            "liquidity": _to_float(market.get("liquidity")),
        })
        
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(["premium_cents", "volume_24h"], ascending=[False, False]).reset_index(drop=True)
    return df


print("Polymarket Analyst helpers initialized, including Disputed Market Analysis.")

Polymarket Analyst helpers initialized, including Disputed Market Analysis.


In [5]:
# Refinements: safer edge math, URL construction, Kelly sizing, report helpers, and local keyword filtering.

def filter_markets_by_text(
    markets_df: pd.DataFrame,
    query: str,
    columns: Sequence[str] = ("event_title", "question", "market_slug", "event_slug"),
) -> pd.DataFrame:
    """Filter normalized markets to rows where query appears in selected text columns."""
    if markets_df.empty or not query:
        return markets_df.copy()
    available_cols = [col for col in columns if col in markets_df.columns]
    if not available_cols:
        return markets_df.copy()
    text = markets_df[available_cols].fillna("").astype(str).agg(" ".join, axis=1)
    return markets_df[text.str.contains(query, case=False, regex=False, na=False)].copy()


def format_market_url(record: Any, *, prefer_event: bool = True) -> str:
    """Return a direct Polymarket URL from a dict/Series-like market or event record.

    Prefer explicit URL fields when present, then construct event or market URLs from common slug fields.
    Returns 'URL unavailable' if no reliable slug/URL is available.
    """
    if record is None:
        return "URL unavailable"
    if isinstance(record, pd.Series):
        data = record.to_dict()
    elif isinstance(record, dict):
        data = record
    else:
        data = {name: getattr(record, name) for name in dir(record) if not name.startswith("_")}

    for key in ("url", "market_url", "event_url"):
        value = data.get(key)
        if isinstance(value, str) and value.startswith("http"):
            return value

    event_slug = data.get("event_slug") or data.get("eventSlug")
    market_slug = data.get("market_slug") or data.get("marketSlug") or data.get("slug")
    if prefer_event and event_slug:
        return f"https://polymarket.com/event/{event_slug}"
    if market_slug:
        return f"https://polymarket.com/market/{market_slug}"
    if event_slug:
        return f"https://polymarket.com/event/{event_slug}"
    return "URL unavailable"


def fractional_kelly(
    fair_prob: float,
    market_price: float,
    *,
    fraction: float = 0.25,
    min_price: float = 0.001,
    max_price: float = 0.999,
) -> Dict[str, float]:
    """Compute full and fractional Kelly for a binary $1 payout YES share.

    Returns a dict with full_kelly, fractional_kelly, edge_per_share, and roi_on_cost.
    Invalid prices or non-positive edges return zero Kelly sizing and NaN ROI where appropriate.
    """
    p = _to_float(fair_prob)
    price = _to_float(market_price)
    if pd.isna(p) or pd.isna(price) or not (0 <= p <= 1) or not (min_price <= price <= max_price):
        return {"full_kelly": 0.0, "fractional_kelly": 0.0, "edge_per_share": np.nan, "roi_on_cost": np.nan}

    edge = p - price
    roi = edge / price
    full = max(edge / (1 - price), 0.0)
    return {
        "full_kelly": float(full),
        "fractional_kelly": float(max(fraction, 0.0) * full),
        "edge_per_share": float(edge),
        "roi_on_cost": float(roi),
    }


def calculate_binary_edge(
    markets_df: pd.DataFrame,
    probability_estimates: Dict[str, float],
    *,
    price_col: str = "yes_price",
    market_key: str = "market_slug",
    kelly_fraction: float = 0.25,
    min_price: float = 0.001,
    max_price: float = 0.999,
) -> pd.DataFrame:
    """Compare external fair probabilities to YES prices with robust ROI and Kelly sizing.

    probability_estimates maps `market_key` values to fair YES probabilities in [0, 1]. Rows outside the
    valid price range keep edge_per_share but receive NaN ROI/Kelly because sizing math is unstable at exact
    0/1 prices and often indicates stale, suspended, or unfillable markets.
    """
    if markets_df.empty:
        return markets_df.copy()

    analysis = markets_df.copy()
    analysis["fair_prob"] = pd.to_numeric(analysis[market_key].map(probability_estimates), errors="coerce").clip(0, 1)
    analysis["market_price"] = pd.to_numeric(analysis[price_col], errors="coerce")
    valid_price = analysis["market_price"].between(min_price, max_price, inclusive="both")
    analysis["edge_per_share"] = analysis["fair_prob"] - analysis["market_price"]
    analysis["roi_on_cost"] = np.where(valid_price, analysis["edge_per_share"] / analysis["market_price"], np.nan)
    analysis["kelly_fraction_binary"] = np.where(
        valid_price,
        ((analysis["fair_prob"] - analysis["market_price"]) / (1 - analysis["market_price"])).clip(lower=0),
        np.nan,
    )
    analysis["fractional_kelly"] = analysis["kelly_fraction_binary"] * max(kelly_fraction, 0.0)
    analysis["price_quality_flag"] = np.where(valid_price, "ok", "invalid_or_stale_for_sizing")
    analysis["polymarket_url"] = analysis.apply(format_market_url, axis=1) if not analysis.empty else []
    return analysis.sort_values("edge_per_share", ascending=False, na_position="last")


def format_analysis_bullets(
    markets_df: pd.DataFrame,
    *,
    max_rows: int = 5,
    include_edge: bool = True,
) -> List[str]:
    """Format top market rows as parent-ready bullets with required Polymarket URLs."""
    if markets_df.empty:
        return ["No matching markets found."]
    bullets: List[str] = []
    for _, row in markets_df.head(max_rows).iterrows():
        name = row.get("question") or row.get("event_title") or row.get("market_slug") or "Unnamed market"
        url = row.get("polymarket_url") or format_market_url(row)
        parts = [f"{name} ({url})"]
        for col, label in (("yes_price", "YES"), ("fair_prob", "fair"), ("edge_per_share", "edge"), ("liquidity", "liq"), ("volume24hr", "24h vol")):
            if col in row and pd.notna(row[col]):
                value = float(row[col])
                parts.append(f"{label}={value:.3f}" if abs(value) <= 1 else f"{label}={value:,.0f}")
        if include_edge and "fractional_kelly" in row and pd.notna(row["fractional_kelly"]):
            parts.append(f"0.25x Kelly={float(row['fractional_kelly']):.2%}")
        bullets.append("; ".join(parts))
    return bullets

print("Refinement helpers loaded: filtering, URL formatting, Kelly sizing, edge analysis, and bullet formatting.")

Refinement helpers loaded: filtering, URL formatting, Kelly sizing, edge analysis, and bullet formatting.


## Use cases and reusable workflows

Use this sub-agent for concrete Polymarket analysis tasks such as:

1. **Market discovery** — “Find active AI, election, Fed, crypto, or sports markets with meaningful volume/liquidity.”
2. **Odds interpretation** — “Summarize current implied probabilities and liquidity for this event or slug.”
3. **Edge analysis** — “Compare my model probability to Polymarket’s YES price and rank possible mispricings.”
4. **Position sizing** — “Suggest conservative fractional Kelly allocation weights for identified positive edges.”
5. **Proposed/disputed resolution screening** — “Find markets with active proposed resolutions and calculate the potential settlement/arbitrage premium.”
6. **Portfolio screening** — “Filter markets by minimum liquidity, ending date, volume, price range, and topic.”
7. **Strategy simulation** — “Given historical snapshots and resolutions, test a threshold strategy before considering any real-world action.”
8. **Risk review** — “Explain caveats: binary payout profile, resolution risk, spread/slippage, stale prices, correlated markets, and sample-size limits.”

How to use the local functions:

- Run the helper definition cells first. Treat them as this sub-agent’s local notebook tools.
- For discovery, call `fetch_events(...)`, normalize with `normalize_events` / `normalize_markets`, optionally filter with `filter_markets_by_text`, then display `summarize_market_universe`.
- For edge and sizing, pass a `{market_slug: fair_probability}` mapping to `calculate_binary_edge`; inspect `edge_per_share`, `roi_on_cost`, `kelly_fraction_binary`, and `fractional_kelly`.
- For parent-ready summaries, call `format_analysis_bullets` so each market line includes a direct Polymarket URL.
- For proposed-resolution tasks, call `fetch_markets_direct` and `build_proposed_markets_table`.
- For backtests, call `backtest_threshold_strategy` with a DataFrame containing `market_price`, `fair_prob`, and `resolved_yes`.

The examples below are intentional reusable examples: they use public read-only endpoints or synthetic data, never place trades, and show expected input/output shapes.

In [6]:
# Example 1: discover active markets by keyword/topic.
# Change QUERY to a topic such as "AI", "election", "Fed", "Bitcoin", or a specific event name.
QUERY = "AI"

try:
    events = fetch_events(search=QUERY, active=True, closed=False, limit=25)
    events_df = normalize_events(events)
    markets_df_raw = normalize_markets(events)
    markets_df = filter_markets_by_text(markets_df_raw, QUERY)
    if markets_df.empty:
        markets_df = markets_df_raw
        print("No local text matches found; showing broad API results instead.")
    print(
        f"Fetched {len(events)} events and normalized {len(markets_df_raw)} markets for query={QUERY!r}; "
        f"showing {len(markets_df)} locally matched/broad markets."
    )
    display(summarize_market_universe(markets_df).head(10))
except Exception as exc:
    print(f"Live API example failed: {type(exc).__name__}: {exc}")
    print("This can happen due to network/API availability. Helper functions remain usable with provided data.")

Fetched 25 events and normalized 891 markets for query='AI'; showing 28 locally matched/broad markets.


,event_title,question,yes_price,no_price,volume24hr,volume,liquidity,end_date,market_slug
0,Strait of Hormuz traffic returns to normal by ...,Strait of Hormuz traffic returns to normal by ...,0.0065,0.9935,1.880108e+06,2.892465e+07,6.189820e+05,2026-05-31T00:00:00Z,strait-of-hormuz-traffic-returns-to-normal-by-...
1,Iran closes its airspace by...?,Iran closes its airspace by May 24?,0.0000,1.0000,1.008999e+06,1.543002e+07,NaN,2026-05-24T00:00:00Z,iran-closes-its-airspace-by-may-24
2,Democratic Presidential Nominee 2028,Will Gina Raimondo win the 2028 Democratic pre...,0.0075,0.9925,8.209677e+05,3.349268e+07,1.660324e+06,2028-11-07T00:00:00Z,will-gina-raimondo-win-the-2028-democratic-pre...
3,California Governor Election Winner,Will Elaine Culotti win the California Governo...,0.0015,0.9985,3.507643e+05,8.623871e+05,1.912916e+05,2026-11-03T00:00:00Z,will-elaine-culotti-win-the-california-governo...
4,Iran closes its airspace by...?,Iran closes its airspace by May 27?,0.0255,0.9745,2.920783e+05,1.895270e+06,5.020410e+04,2026-05-27T00:00:00Z,iran-closes-its-airspace-by-may-29
5,2026 FIFA World Cup Winner,Will Spain win the 2026 FIFA World Cup?,0.1725,0.8275,2.005424e+05,2.486060e+07,7.367061e+05,2026-07-20T00:00:00Z,will-spain-win-the-2026-fifa-world-cup-963
6,Brazil Presidential Election,Will Jair Bolsonaro win the 2026 Brazilian pre...,0.0065,0.9935,1.669378e+05,4.045669e+06,3.206509e+05,2026-10-04T00:00:00Z,will-jair-bolsonaro-win-the-2026-brazilian-pre...
7,Iran closes its airspace by...?,Iran closes its airspace by May 31?,0.1200,0.8800,1.592961e+05,6.676378e+06,3.276643e+04,2026-05-31T00:00:00Z,iran-closes-its-airspace-by-may-31-434-443-672...
8,Iran closes its airspace by...?,Iran closes its airspace by June 30?,0.3230,0.6770,1.223517e+05,1.951380e+06,2.399114e+04,2026-06-30T00:00:00Z,iran-closes-its-airspace-by-june-30-432-786-46...
9,Brazil Presidential Election,Will Ronaldo Caiado win the 2026 Brazilian pre...,0.0175,0.9825,7.606091e+04,3.365906e+06,1.444200e+05,2026-10-04T00:00:00Z,will-ronaldo-caiado-win-the-2026-brazilian-pre...


In [7]:
# Example 2: compare external fair probabilities to current YES prices and format parent-ready bullets.
# In a real task, probability_estimates should come from a model, forecast, or user-provided assumptions.

if "markets_df" in globals() and not markets_df.empty and "market_slug" in markets_df.columns:
    liquid_markets = markets_df.dropna(subset=["market_slug", "yes_price"]).copy()
    liquid_markets = liquid_markets[liquid_markets["yes_price"].between(0.001, 0.999, inclusive="both")]
    candidate_markets = liquid_markets.head(3).copy()
    probability_estimates = {
        row.market_slug: min(max(float(row.yes_price) + 0.03, 0.01), 0.99)
        for row in candidate_markets.itertuples()
    }
    edge_df = calculate_binary_edge(candidate_markets, probability_estimates, kelly_fraction=0.25)
    display(edge_df[["question", "market_slug", "polymarket_url", "yes_price", "fair_prob", "edge_per_share", "roi_on_cost", "kelly_fraction_binary", "fractional_kelly", "price_quality_flag"]])
    print("\nExample parent-ready bullets:")
    for bullet in format_analysis_bullets(edge_df, max_rows=3):
        print(f"- {bullet}")
else:
    print("Run Example 1 first or provide a markets_df DataFrame.")

,question,market_slug,polymarket_url,yes_price,fair_prob,edge_per_share,roi_on_cost,kelly_fraction_binary,fractional_kelly,price_quality_flag
0,Will Spain win the 2026 FIFA World Cup?,will-spain-win-the-2026-fifa-world-cup-963,https://polymarket.com/event/2026-fifa-world-c...,0.1725,0.2025,0.03,0.173913,0.036254,0.009063,ok
126,Will Gina Raimondo win the 2028 Democratic pre...,will-gina-raimondo-win-the-2028-democratic-pre...,https://polymarket.com/event/democratic-presid...,0.0075,0.0375,0.03,4.000000,0.030227,0.007557,ok
516,Strait of Hormuz traffic returns to normal by ...,strait-of-hormuz-traffic-returns-to-normal-by-...,https://polymarket.com/event/strait-of-hormuz-...,0.0065,0.0365,0.03,4.615385,0.030196,0.007549,ok



Example parent-ready bullets:
- Will Spain win the 2026 FIFA World Cup? (https://polymarket.com/event/2026-fifa-world-cup-winner-595); YES=0.172; fair=0.202; edge=0.030; liq=736,706; 24h vol=200,542; 0.25x Kelly=0.91%
- Will Gina Raimondo win the 2028 Democratic presidential nomination? (https://polymarket.com/event/democratic-presidential-nominee-2028); YES=0.007; fair=0.037; edge=0.030; liq=1,660,324; 24h vol=820,968; 0.25x Kelly=0.76%
- Strait of Hormuz traffic returns to normal by end of May? (https://polymarket.com/event/strait-of-hormuz-traffic-returns-to-normal-by-end-of-may); YES=0.006; fair=0.036; edge=0.030; liq=618,982; 24h vol=1,880,108; 0.25x Kelly=0.75%


In [8]:
# Example 3: toy threshold-strategy backtest on synthetic historical snapshots.
# This demonstrates the expected input shape for user-provided historical data.

synthetic_snapshots = pd.DataFrame(
    {
        "market_slug": ["m1", "m2", "m3", "m4", "m5"],
        "timestamp": pd.date_range("2024-01-01", periods=5, freq="D"),
        "market_price": [0.40, 0.55, 0.30, 0.70, 0.20],
        "fair_prob": [0.50, 0.57, 0.42, 0.68, 0.33],
        "resolved_yes": [1, 0, 1, 1, 0],
    }
)

backtest_result = backtest_threshold_strategy(synthetic_snapshots, min_edge=0.08, stake=10.0)
print({k: v for k, v in backtest_result.items() if k != "trades"})
display(backtest_result["trades"])

{'n_trades': 3, 'total_staked': 30.0, 'pnl': 28.333333333333336, 'roi': 0.9444444444444445, 'win_rate': 0.6666666666666666, 'avg_edge': 0.11666666666666665}


,market_slug,timestamp,market_price,fair_prob,resolved_yes,edge,stake,shares,payout,pnl
0,m1,2024-01-01,0.4,0.50,1,0.10,10.0,25.000000,25.000000,15.000000
2,m3,2024-01-03,0.3,0.42,1,0.12,10.0,33.333333,33.333333,23.333333
4,m5,2024-01-05,0.2,0.33,0,0.13,10.0,50.000000,0.000000,-10.000000


In [9]:
# Example 4: Disputed/Proposed Market Analysis
# This finds markets that have a proposed resolution but are not yet settled, 
# calculating the potential arbitrage premium.

try:
    # Fetch liquid markets
    liquid_markets = fetch_markets_direct(min_volume24h=250.0)
    proposed_df = build_proposed_markets_table(liquid_markets)

    print(f"Total liquid markets fetched: {len(liquid_markets)}")
    print(f"Active proposed markets found: {len(proposed_df)}")

    if not proposed_df.empty:
        display(proposed_df[['question', 'implied_proposal', 'last_price', 'premium_cents', 'volume_24h', 'url']].head(10))
    else:
        print("No active proposed markets found at this time.")
except Exception as exc:
    print(f"Disputed market analysis failed: {type(exc).__name__}: {exc}")

Total liquid markets fetched: 5353
Active proposed markets found: 92


,question,implied_proposal,last_price,premium_cents,volume_24h,url
0,Set Handicap: Baptiste (-1.5) vs Wang (+1.5),Yes,0.500,50.0,4235.838078,https://polymarket.com/market/wta-baptist-wan-...
1,Counter-Strike: LPH Gaming vs CSGOPOSITIVE - M...,Yes,0.550,45.0,5558.314061,https://polymarket.com/market/cs2-lph-csgopo-2...
2,Will there be a run scored in the first inning...,No,0.360,36.0,3218.439773,https://polymarket.com/market/mlb-cin-nym-2026...
3,Iran closes its airspace by June 30?,No,0.319,31.9,122351.650036,https://polymarket.com/market/iran-closes-its-...
4,Iran closes its airspace by June 15?,No,0.295,29.5,66459.249103,https://polymarket.com/market/iran-closes-its-...
5,Exact Score: FC Nantes 0 - 3 Toulouse FC?,Yes,0.750,25.0,1861.754122,https://polymarket.com/market/fl1-nan-tou-2026...
6,Exact Score: FC Nantes 0 - 0 Toulouse FC?,No,0.250,25.0,452.378959,https://polymarket.com/market/fl1-nan-tou-2026...
7,Iran closes its airspace by May 31?,No,0.132,13.2,159296.130675,https://polymarket.com/market/iran-closes-its-...
8,Epstein client list released by June 30?,No,0.084,8.4,2721.612437,https://polymarket.com/market/epstein-client-l...
9,Ruben Rocha out as Governor of Sinaloa by May 31?,No,0.080,8.0,6372.032950,https://polymarket.com/market/ruben-rocha-out-...
